In [1]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate bert-score sacrebleu unbabel-comet bert-score -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 185.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 162.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-language 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you 

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [2]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

In [3]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

# SILABIFICADOR

In [4]:
# =========================
# DEFINICIÓN DEL SISTEMA
# =========================

VOWELS = ["a", "e", "i", "o"]
DIGRAPHS = ["ch", "sh", "ts", "ty"]
NASALS = ["n", "m"]


# =========================
# TOKENIZADOR
# =========================

def tokenize(word):
    """
    Regla: los dígrafos son una sola unidad fonológica
    """
    tokens = []
    i = 0

    while i < len(word):
        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens


# =========================
# SILABIFICADOR V3.1
# =========================

def silabificar_v3(word):
    tokens = tokenize(word)
    syllables = []
    i = 0

    while i < len(tokens):
        syllable = ""

        # -------------------------
        # Regla: inicio consonántico opcional
        # -------------------------
        if tokens[i] not in VOWELS:
            syllable += tokens[i]
            i += 1

        # -------------------------
        # Regla: núcleo vocálico obligatorio
        # -------------------------
        if i < len(tokens) and tokens[i] in VOWELS:
            syllable += tokens[i]
            i += 1

            #  Regla 9: agrupar vocales consecutivas (diptongos)
            while i < len(tokens) and tokens[i] in VOWELS:
                syllable += tokens[i]
                i += 1

        else:
            syllables.append(syllable)
            break

        # -------------------------
        # Regla: coda nasal opcional (n o m)
        # -------------------------
        if i < len(tokens) and tokens[i] in NASALS:
            if i + 1 < len(tokens) and tokens[i+1] not in VOWELS:
                syllable += tokens[i]
                i += 1

        syllables.append(syllable)

    # =========================
    # POST-PROCESAMIENTO
    # =========================

    # -------------------------
    # Regla 6: evitar nasal final en palabra
    # -------------------------
    if len(syllables) > 1:
        last = syllables[-1]

        if last in NASALS:
            syllables[-2] += last
            syllables = syllables[:-1]

    # -------------------------
    # Regla 7: evitar sílabas sin vocal
    # -------------------------
    cleaned = []
    for syl in syllables:
        if any(v in syl for v in VOWELS):
            cleaned.append(syl)
        else:
            if cleaned:
                cleaned[-1] += syl
            else:
                cleaned.append(syl)

    return cleaned


# =========================
# TEST
# =========================

words = [
    "maranke",
    "ampeji",
    "inki",
    "inchato",
    "wishawe",
    "ratekashamaira",
    "wanoparirin"
]

for w in words:
    print(w, "→", silabificar_v3(w))

maranke → ['ma', 'ran', 'ke']
ampeji → ['am', 'pe', 'ji']
inki → ['in', 'ki']
inchato → ['in', 'cha', 'to']
wishawe → ['wi', 'sha', 'we']
ratekashamaira → ['ra', 'te', 'ka', 'sha', 'mai', 'ra']
wanoparirin → ['wa', 'no', 'pa', 'ri', 'rin']


In [5]:
def silabificar_texto(texto):
    """
    Regla: el silabificador trabaja a nivel de palabra,
    por lo que primero se separa la frase en palabras.
    """
    palabras = texto.split()
    resultado = []

    for palabra in palabras:
        silabas = silabificar_v3(palabra)
        resultado.extend(silabas)

    return resultado

In [6]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificar_v3(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))

# ==============================================================================
# FILTRAR TOKENS ÚTILES DEL SILABIFICADOR (UMBRAL >= 20)
# ==============================================================================
nuevos_tokens = [token for token, count in subword_counter.items() if count >= 20]
print("Cantidad de nuevos tokens a agregar (freq >= 20):", len(nuevos_tokens))


SUBWORDS MÁS FRECUENTES:
[('ra', 2860), ('ki', 2243), ('ke', 2043), ('ja', 2017), ('ti', 2000), ('i', 1866), ('a', 1580), ('we', 1426), ('ka', 1389), ('ri', 1276), ('ma', 1165), ('na', 1141), ('mia', 1035), ('ea', 940), ('en', 794), ('yo', 759), ('ya', 711), ('ba', 621), ('ko', 592), ('o', 555), ('ta', 545), ('pa', 535), ('min', 506), ('bi', 505), ('no', 487), ('to', 478), ('ni', 451), ('nai', 397), ('bo', 327), ('kon', 315), ('nan', 312), ('kai', 310), ('be', 295), ('rin', 284), ('ne', 276), ('sai', 272), ('pi', 269), ('noa', 246), ('tom', 243), ('kin', 243), ('me', 241), ('ro', 230), ('kan', 229), ('shi', 225), ('nin', 224), ('pan', 220), ('man', 220), ('te', 218), ('yoi', 206), ('non', 205)]
Cantidad de nuevos tokens a agregar (freq >= 20): 146


***Creación del dataset***

In [7]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_model["source"] = df_model["source"].apply(
    lambda x: "translate Ashaninka to Spanish: " + x
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 5776
})


***Tokenización***

In [8]:
## Tokenizer mT5

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# ==============================================================================
# AGREGAR TOKENS AL TOKENIZER (Aquí se agregan los nuevos tokens al tokenizador)
# ==============================================================================
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados con éxito:", tokens_agregados)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Nuevos tokens reales a agregar: 31
Tokens agregados con éxito: 31


In [9]:
# Prompt ya prependeado en la celda de creacion del dataset
print("Prompt ya configurado.")


Prompt ya configurado.


In [10]:
#Función de tokenización

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [11]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

#Aplicación de tokenización

tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/5776 [00:00<?, ? examples/s]

In [12]:
# ==============================================================================
# DIAGNÓSTICO DE TOKENIZACIÓN (VERIFICACIÓN DE ESTRUCTURA Y EOS)
# ==============================================================================
ejemplo_idx = 0
input_ids = tokenized_dataset[ejemplo_idx]["input_ids"]
label_ids = tokenized_dataset[ejemplo_idx]["labels"]

print("INPUT IDS:", input_ids)
print("INPUT DECODED:", tokenizer.decode(input_ids))
print("---")
print("LABEL IDS (con -100 reemplazado por pad para decodificar):", label_ids.tolist() if hasattr(label_ids, 'tolist') else label_ids)
# Reemplazar -100 por pad_token_id (0) para decodificar
labels_list = label_ids.tolist() if hasattr(label_ids, 'tolist') else label_ids
labels_decodificables = [(l if l != -100 else tokenizer.pad_token_id) for l in labels_list]
print("LABEL DECODED:", tokenizer.decode(labels_decodificables))


INPUT IDS: tensor([37194,   298, 20427, 19470,   288,   259, 29037,   267,   432, 31556,
          658, 10711,  7238,   259,  3487, 63741,     1,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])
INPUT DECODED: translate Ashaninka to Spanish: jatian miabicho ika ikon</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
---
LABEL IDS (con -100 reemplazado por pad para decodificar): [259, 1401, 30763, 23282, 358, 259, 250119, 707, 25325, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Modelo

In [13]:
##Llamar al modelo

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

MT5ForConditionalGeneration(
  (shared): Embedding(250131, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250131, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [14]:
##Parámetros para entrenamiento

from transformers import TrainingArguments, Trainer

# ==============================================================================
# PARÁMETROS FASE 1 (ENTRENAMIENTO INICIAL - OPTIMIZADO PARA AHORRO DE TIEMPO):
# - per_device_train_batch_size=8: Mayor tamaño de lote para acelerar procesamiento.
# - num_train_epochs=6: Reducido de 10 a 6 para ahorrar recursos de Colab (tiempo y créditos).
# - learning_rate=5e-4: Incrementado para acelerar convergencia con lote más grande.
# - warmup_steps=100: Calentamiento de aprendizaje breve para estabilidad.
# - weight_decay=0.01: Regularización estándar.
# - logging_steps=200: Registro de pérdidas frecuente.
# ==============================================================================
training_args = TrainingArguments(
    output_dir="./results_mt5",
    per_device_train_batch_size=8,
    num_train_epochs=6,
    learning_rate=5e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [15]:
##Entrenar el modelo

trainer.train()

Step,Training Loss
200,20.380700
400,0.918500
600,0.797000
800,0.544500
1000,0.468100
1200,0.432700
1400,0.428700
1600,0.396000
1800,0.377900
2000,0.370100


TrainOutput(global_step=4332, training_loss=1.3345121337876535, metrics={'train_runtime': 289.2604, 'train_samples_per_second': 119.809, 'train_steps_per_second': 14.976, 'total_flos': 2290673120182272.0, 'train_loss': 1.3345121337876535, 'epoch': 6.0})

In [16]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4/tokenizer.json')

In [17]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO Y TOKENIZER (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 y tokenizer desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo y tokenizer desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 y tokenizer desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [18]:
def traducir(texto):
    texto_con_prompt = "translate Ashaninka to Spanish: " + texto
    inputs = tokenizer(texto_con_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        num_beams=4,           #  beam search
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Validacion (Val.csv)

In [19]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [20]:
#Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

from datasets import Dataset
df_val_model = df_val[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_val_model["source"] = df_val_model["source"].apply(
    lambda x: "translate Ashaninka to Spanish: " + x
)
val_dataset = Dataset.from_pandas(df_val_model)
print("Dataset de validación cargado con éxito:", val_dataset)


Dataset de validación cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 722
})


In [21]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: cabayoain ea neti atipana
REAL: puedo montar un caballo
PRED: puedo ayudarte
-----
INPUT: tsoaki joa
REAL: quién ha venido
PRED: so y profeso lo
-----
INPUT: tsonbira yoiyamake
REAL: nadie respondió
PRED: so y profeso r
-----
INPUT: napomea ea meniwe
REAL: dame la mitad
PRED: qué pasó aquí
-----
INPUT: bakishra en oinkasai
REAL: espero verla mañana
PRED: no puedo hablar conmigo
-----


# Métricas

In [22]:
#Preparar predicciones

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [23]:
#BLEU

import evaluate
bleu = evaluate.load("bleu")

print(bleu.compute(predictions=preds, references=refs))

{'bleu': 0.0, 'precisions': [0.10371032632990612, 0.018481848184818482, 0.0012610340479192938, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0122171945701357, 'translation_length': 2237, 'reference_length': 2210}


In [24]:
#CHRF
chrf = evaluate.load("chrf")

print(chrf.compute(predictions=preds, references=refs))

{'score': 13.512074810018213, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [25]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  # ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print(scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/437 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') th

-1.1924707586580365


In [26]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.06217011353196334}


In [27]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Promedio): 0.7287812584655107
BERTScore Recall (Promedio): 0.7265358574667796
BERTScore F1 (Promedio): 0.7273679870152407


In [33]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL (FASE 2 - OPTIMIZADO):
# 1. learning_rate=8e-5: Tasa de aprendizaje moderadamente más baja para ajuste fino.
# 2. num_train_epochs=3: Reducido para optimizar tiempo de ejecución.
# 3. per_device_train_batch_size=8: Lote consistente con Fase 1.
# 4. warmup_steps=50: Calentamiento corto.
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=6,
    learning_rate=8e-5,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

In [34]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [35]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [36]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) y tokenizer para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Embedding(250131, 512)

In [37]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [38]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
200,0.301400
400,0.292600
600,0.296200
800,0.281900
1000,0.272900
1200,0.264000
1400,0.269700
1600,0.256200
1800,0.250400
2000,0.249200


TrainOutput(global_step=4332, training_loss=0.2595913236271091, metrics={'train_runtime': 291.7173, 'train_samples_per_second': 118.8, 'train_steps_per_second': 14.85, 'total_flos': 2290673120182272.0, 'train_loss': 0.2595913236271091, 'epoch': 6.0})

In [39]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mT5_ashaninka_v4_retrained/tokenizer.json')

In [40]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: cabayoain ea neti atipana
REAL: puedo montar un caballo
PRED: puedo hacerlo ahora
-----
INPUT: tsoaki joa
REAL: quién ha venido
PRED: esa bes a su hermana
-----
INPUT: tsonbira yoiyamake
REAL: nadie respondió
PRED: acaso n universitarios
-----
INPUT: napomea ea meniwe
REAL: dame la mitad
PRED: so y profeso tros
-----
INPUT: bakishra en oinkasai
REAL: espero verla mañana
PRED: mañana estar mañana
-----


In [41]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [42]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.0,
 'precisions': [0.12412831241283125,
  0.026592022393282014,
  0.002828854314002829,
  0.0],
 'brevity_penalty': 0.9729436591450357,
 'length_ratio': 0.9733031674208145,
 'translation_length': 2151,
 'reference_length': 2210}

In [43]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 15.747953342357256, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [44]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.5663583274941035


In [45]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


METEOR (Validación - Reentrenado): {'meteor': 0.07568508842146589}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [46]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Validación - Reentrenado - Promedio): 0.7366727700358943
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7331225042362953
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7345740982535143


In [47]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [48]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [49]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 723
})


In [50]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [51]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.0,
 'precisions': [0.12471131639722864,
  0.02565880721220527,
  0.002777777777777778,
  0.0],
 'brevity_penalty': 0.9921785539476508,
 'length_ratio': 0.9922089825847846,
 'translation_length': 2165,
 'reference_length': 2182}

In [52]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 15.73294487362998, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [53]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.5666819892483628


In [54]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


METEOR: {'meteor': 0.07335093789189003}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [55]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Promedio): 0.7334385139978443
BERTScore Recall (Promedio): 0.729903553119174
BERTScore F1 (Promedio): 0.7313479805883035
